In [3]:
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

print("1. Loading data...")
df = pd.read_csv("Diabetes.csv")
df = df.drop_duplicates().reset_index(drop=True)

print("2. Defining features and target...")
X = df.drop('diabetes', axis=1)
y = df['diabetes']

# Match exact columns from your knowledge base
numeric_features = ['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level', 'blood_glucose_level']
categorical_features = ['smoking_history']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_features)
    ])

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=10),
    'Support Vector Machine': SVC(kernel='rbf', probability=True, random_state=42)
}

print("3. Training and saving models...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    filename = f"{name.replace(' ', '_').lower()}.pkl"
    joblib.dump(pipeline, filename)
    print(f"   ✅ Saved: {filename}")

# Save feature names and background data for SHAP
feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
joblib.dump(feature_names, "feature_names.pkl")

background_data = X_train.sample(100, random_state=42)
joblib.dump(background_data, "background_data.pkl")

print("🎉 All models and artifacts saved successfully!")

1. Loading data...
2. Defining features and target...
3. Training and saving models...
   ✅ Saved: logistic_regression.pkl
   ✅ Saved: k-nearest_neighbors.pkl
   ✅ Saved: support_vector_machine.pkl
🎉 All models and artifacts saved successfully!
